Stratified Random Sampling Based on Sentiment Polarity

This gives us:

*   Balanced emotional content
* Avoids over-representation of positive or negative sentences





*   Matches the MAMS design of multiple sentiments in one sentence






This is the same principle used in GoEmotions, where they balanced sentiment categories during data selection.

In [ ]:
from google.colab import files
uploaded = files.upload()   # Upload train.xml


Saving train.xml to train (1).xml


In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

xml_path = "train.xml"

tree = ET.parse(xml_path)
root = tree.getroot()

rows = []

for sent in root.findall("sentence"):
    text_el = sent.find("text")
    if text_el is None:
        continue

    text = text_el.text.strip()

    aspects_el = sent.find("aspectCategories")
    categories = []
    polarities = []

    if aspects_el is not None:
        for ac in aspects_el.findall("aspectCategory"):
            categories.append(ac.get("category"))
            polarities.append(ac.get("polarity"))

    num_aspects = len(polarities)
    pos_count = sum(p == "positive" for p in polarities)
    neg_count = sum(p == "negative" for p in polarities)
    neu_count = sum(p == "neutral" for p in polarities)

    # Classify polarity pattern (for stratified sampling)
    if pos_count > 0 and neg_count > 0:
        polarity_type = "mixed"
    elif pos_count > 0 and neg_count == 0 and neu_count == 0:
        polarity_type = "pos_only"
    elif neg_count > 0 and pos_count == 0 and neu_count == 0:
        polarity_type = "neg_only"
    elif neu_count > 0 and pos_count == 0 and neg_count == 0:
        polarity_type = "neu_only"
    else:
        polarity_type = "other"

    # Token length
    tokens = text.split()
    length_tokens = len(tokens)
    if length_tokens <= 10:
        length_bucket = "short"
    elif length_tokens <= 20:
        length_bucket = "medium"
    else:
        length_bucket = "long"

    rows.append({
        "text": text,
        "categories": categories,
        "polarities": polarities,
        "num_aspects": num_aspects,
        "pos_count": pos_count,
        "neg_count": neg_count,
        "neu_count": neu_count,
        "polarity_type": polarity_type,
        "length_tokens": length_tokens,
        "length_bucket": length_bucket,
    })

df = pd.DataFrame(rows)
df.head()


### This will correctly extract:

# sentence text

# aspect categories (e.g., food, staff, place)

# polarity for each aspect (positive/neutral/negative)

# counts of pos/neg/neu

# length bucket

# polarity type (for stratification) ##

,text,categories,polarities,num_aspects,pos_count,neg_count,neu_count,polarity_type,length_tokens,length_bucket
0,It might be the best sit down food I've had in...,"[food, place]","[positive, neutral]",2,1,0,1,other,34,long
1,Hostess was extremely accommodating when we ar...,"[staff, miscellaneous]","[positive, neutral]",2,1,0,1,other,13,medium
2,We were a couple of minutes late for our reser...,"[miscellaneous, staff]","[neutral, negative]",2,0,1,1,other,27,long
3,"Though the service might be a little slow, the...","[service, staff]","[negative, positive]",2,1,1,0,mixed,13,medium
4,Although we arrived at the restaurant 10 min l...,"[staff, miscellaneous]","[negative, neutral]",2,0,1,1,other,18,medium


In [ ]:
df["polarity_type"].value_counts()
# Inspect distribution

,count
polarity_type,
other,2625
mixed,524


In [ ]:
# Stratified sampling: 300 sentences

# Goal:

# 100 mixed

# 100 pos_only

# 100 neg_only
# (or adjust to match availability)

target_per_type = {
    "mixed": 100,
    "pos_only": 100,
    "neg_only": 100,
}

def stratified_sample(df, group_col, n_per_group, random_state=42):
    samples = []
    for g, n in n_per_group.items():
        group_df = df[df[group_col] == g]
        if group_df.empty:
            continue
        if len(group_df) <= n:
            samples.append(group_df)
        else:
            samples.append(group_df.sample(n, random_state=random_state))
    return pd.concat(samples).sample(frac=1, random_state=random_state).reset_index(drop=True)

sample_300 = stratified_sample(df, "polarity_type", target_per_type)
len(sample_300), sample_300["polarity_type"].value_counts()


(100,
 polarity_type
 mixed    100
 Name: count, dtype: int64)

In [ ]:
# save final for annotation
output_path = "mams_300_for_annotation.csv"
sample_300.to_csv(output_path, index=False)

from google.colab import files
files.download(output_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>